# 12 Demo - Ingesting JSON files with Databricks

**Course:** Data Ingestion with Lakeflow Connect  
**Note:** Because direct video scraping isn't accessible from the embedded player, this practice notebook reproduces the exact structure and topics shown in the demo (`B. Overview of CTAS with read_files() for Ingestion of JSON files`, `B1. Inspect JSON files`, Base64 decoding, `STRUCT` flattening, `explode()` / `explode_outer()`, and `VARIANT`), using self-contained sample Kafka JSON data so you can run every cell directly in Databricks (including Databricks Free Edition).

## A. Setup Sample Kafka E-Commerce JSON Data (Self-Contained Practice Setup)

In the Databricks Academy demo, raw Kafka event files live at `/Volumes/dbacademy_ecommerce/v01/raw/events-kafka`. Since that volume is specific to the hosted Academy lab, run the Python setup cell below to generate realistic sample Kafka JSON files (with Base64-encoded `key` and `value` fields) in a temporary practice path or your own Unity Catalog Volume.

In [ ]:
%python
import base64, json, os

# Update this path to a Unity Catalog Volume if desired, e.g. '/Volumes/main/default/practice_vol/events-kafka'
# For quick practice without creating a volume, we write to /tmp/dbacademy_practice/events-kafka
sample_dir = "/tmp/dbacademy_practice/events-kafka"
os.makedirs(sample_dir, exist_ok=True)

sample_events = [
    {
        "key": base64.b64encode(b"user_1001").decode("utf-8"),
        "offset": 219255030,
        "partition": 0,
        "timestamp": 1730000001000,
        "topic": "ecommerce_events",
        "value": base64.b64encode(json.dumps({
            "event_id": "evt_001",
            "event_name": "checkout",
            "user_id": "user_1001",
            "device": "Android",
            "geo": {"city": "Visakhapatnam", "state": "AP", "country": "IN"},
            "items": [
                {"item_id": "SKU-101", "item_name": "Wireless Mouse", "quantity": 1, "price": 29.99},
                {"item_id": "SKU-205", "item_name": "USB-C Hub", "quantity": 2, "price": 45.00}
            ]
        }).encode("utf-8")).decode("utf-8")
    },
    {
        "key": base64.b64encode(b"user_1002").decode("utf-8"),
        "offset": 219255031,
        "partition": 0,
        "timestamp": 1730000005000,
        "topic": "ecommerce_events",
        "value": base64.b64encode(json.dumps({
            "event_id": "evt_002",
            "event_name": "page_view",
            "user_id": "user_1002",
            "device": "iOS",
            "geo": {"city": "Hyderabad", "state": "TG", "country": "IN"},
            "items": []
        }).encode("utf-8")).decode("utf-8")
    },
    {
        "key": base64.b64encode(b"user_1003").decode("utf-8"),
        "offset": 219255032,
        "partition": 1,
        "timestamp": 1730000010000,
        "topic": "ecommerce_events",
        "value": base64.b64encode(json.dumps({
            "event_id": "evt_003",
            "event_name": "add_to_cart",
            "user_id": "user_1003",
            "device": "MacOS",
            "geo": {"city": "Bengaluru", "state": "KA", "country": "IN"},
            "items": [
                {"item_id": "SKU-309", "item_name": "Mechanical Keyboard", "quantity": 1, "price": 119.50}
            ],
            "promo_code": "SAVE10"
        }).encode("utf-8")).decode("utf-8")
    }
]

file_path = os.path.join(sample_dir, "events_001.json")
with open(file_path, "w") as f:
    for row in sample_events:
        f.write(json.dumps(row) + "\n")

print(f"Sample Kafka JSON file created at: {file_path}")

## B. Overview of CTAS with `read_files()` for Ingestion of JSON files

### B1. Inspect JSON files

1. Run the next cell to list the JSON files (in the Academy lab, this points to `'/Volumes/dbacademy_ecommerce/v01/raw/events-kafka'`).

In [ ]:
-- Academy Lab Path:
-- LIST '/Volumes/dbacademy_ecommerce/v01/raw/events-kafka'

-- Practice Path:
LIST 'file:/tmp/dbacademy_practice/events-kafka'

2. Run the cell below to view the raw JSON data in the output using `read_files()`. Note the following:
* Each row contains JSON with 6 key/value pairs (`key`, `offset`, `partition`, `timestamp`, `topic`, `value`).
* The `key` and `value` fields are encoded in **Base64**.

In [ ]:
SELECT *
FROM read_files(
  'file:/tmp/dbacademy_practice/events-kafka',
  format => 'json'
);

### B2. Ingest Raw Kafka JSON via CTAS (`CREATE TABLE AS SELECT`)

Ingest the raw JSON records into a Bronze Delta table using `CREATE OR REPLACE TABLE ... AS SELECT * FROM read_files(...)`.

In [ ]:
CREATE OR REPLACE TABLE kafka_events_bronze AS
SELECT *
FROM read_files(
  'file:/tmp/dbacademy_practice/events-kafka',
  format => 'json'
);

SELECT * FROM kafka_events_bronze;

## C. Decoding Base64 Fields (`unbase64()`)

Kafka `key` and `value` payloads are stored as Base64 strings. Use `unbase64()` cast to `STRING` (or `CAST(unbase64(...) AS STRING)`) to decode them into readable text and JSON strings.

In [ ]:
CREATE OR REPLACE TABLE kafka_events_decoded AS
SELECT
  CAST(unbase64(key) AS STRING) AS decoded_key,
  offset,
  partition,
  timestamp,
  topic,
  CAST(unbase64(value) AS STRING) AS decoded_value
FROM kafka_events_bronze;

SELECT * FROM kafka_events_decoded;

## D. Flattening JSON Strings Without `STRUCT` (`:` Colon Operator & `get_json_object()`)

Databricks SQL supports traversing JSON strings directly with the `:` path operator (e.g., `decoded_value:event_name`) without defining an upfront schema.

In [ ]:
SELECT
  decoded_key,
  decoded_value:event_id::STRING AS event_id,
  decoded_value:event_name::STRING AS event_name,
  decoded_value:user_id::STRING AS user_id,
  decoded_value:device::STRING AS device,
  decoded_value:geo.city::STRING AS city,
  decoded_value:geo.state::STRING AS state,
  decoded_value:items AS items_json_str
FROM kafka_events_decoded;

## E. Converting JSON Strings to `STRUCT` (`schema_of_json()` + `from_json()`)

For strong typing, schema enforcement, and nested dot-notation access, infer the JSON schema with `schema_of_json()` and parse the column into a `STRUCT` with `from_json()`.

In [ ]:
-- Step 1: Inspect the inferred DDL schema from a sample JSON payload
SELECT schema_of_json(
  '{"event_id":"evt_001","event_name":"checkout","user_id":"user_1001","device":"Android","geo":{"city":"Visakhapatnam","state":"AP","country":"IN"},"items":[{"item_id":"SKU-101","item_name":"Wireless Mouse","quantity":1,"price":29.99}],"promo_code":"SAVE10"}'
) AS inferred_schema;

In [ ]:
-- Step 2: Parse decoded_value into a strongly typed STRUCT column
CREATE OR REPLACE TABLE kafka_events_struct AS
SELECT
  decoded_key,
  offset,
  partition,
  timestamp,
  from_json(
    decoded_value,
    'STRUCT<device: STRING, event_id: STRING, event_name: STRING, geo: STRUCT<city: STRING, country: STRING, state: STRING>, items: ARRAY<STRUCT<item_id: STRING, item_name: STRING, price: DOUBLE, quantity: BIGINT>>, promo_code: STRING, user_id: STRING>'
  ) AS event_struct
FROM kafka_events_decoded;

-- Flatten nested STRUCT fields using dot notation
SELECT
  decoded_key,
  event_struct.event_id,
  event_struct.event_name,
  event_struct.user_id,
  event_struct.device,
  event_struct.geo.city,
  event_struct.geo.country,
  event_struct.promo_code,
  event_struct.items
FROM kafka_events_struct;

## F. Exploding Arrays: `explode()` vs `explode_outer()`

* `explode(array)` drops rows where the array is empty (`[]`) or `NULL` (such as `page_view` events with no items).
* `explode_outer(array)` preserves all parent rows, returning `NULL` for exploded columns when the array is empty or `NULL`.

In [ ]:
-- explode(): Drops events with empty items array (evt_002 is omitted)
SELECT
  event_struct.event_id,
  event_struct.event_name,
  explode(event_struct.items) AS item
FROM kafka_events_struct;

In [ ]:
-- explode_outer(): Keeps all events, flattening item fields cleanly
SELECT
  event_struct.event_id,
  event_struct.event_name,
  event_struct.user_id,
  exploded_item.item_id,
  exploded_item.item_name,
  exploded_item.quantity,
  exploded_item.price
FROM (
  SELECT
    event_struct,
    explode_outer(event_struct.items) AS exploded_item
  FROM kafka_events_struct
);

## G. Working with the `VARIANT` Data Type (`parse_json()`)

The `VARIANT` data type stores semi-structured JSON natively in Delta tables without rigid upfront schemas—ideal when JSON keys evolve over time (like `promo_code` appearing only on certain events).

In [ ]:
CREATE OR REPLACE TABLE kafka_events_variant AS
SELECT
  decoded_key,
  offset,
  partition,
  timestamp,
  parse_json(decoded_value) AS event_variant
FROM kafka_events_decoded;

-- Query VARIANT fields with colon (:) and cast with ::
SELECT
  decoded_key,
  event_variant:event_id::STRING AS event_id,
  event_variant:event_name::STRING AS event_name,
  event_variant:geo.city::STRING AS city,
  event_variant:promo_code::STRING AS promo_code,
  variant_explode_outer(event_variant:items) AS item_kv
FROM kafka_events_variant;